# Orbital Entanglement Chord Diagram — 250 Synthetic Orbitals

Demonstrates the `OrbitalEntanglement` JS widget on a large system with
**250 orbitals**.  50 orbitals form a strongly-coupled core with high
single-orbital entropies; the remaining 200 have a long tail of weak
entropy and negligible mutual information.

No quantum chemistry calculation is needed — we build a mock dataset
with NumPy and pass raw arrays directly to the widget.

## 1 — Build synthetic data

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

N = 250        # total orbitals
N_CORE = 50    # strongly-coupled subset to highlight

# ── Single-orbital entropies ─────────────────────────────────────────
# Three regimes interleaved across the orbital indices:
#   • ~50 "core" orbitals with high entropy (near ln 4)
#   • ~60 "medium" orbitals with moderate entropy
#   • ~140 "spectator" orbitals with a long decaying tail
s1 = np.zeros(N)

all_indices = rng.permutation(N)
core_idx = np.sort(all_indices[:N_CORE])
medium_idx = np.sort(all_indices[N_CORE : N_CORE + 60])
tail_idx = np.sort(all_indices[N_CORE + 60 :])

s1[core_idx] = rng.beta(2.5, 1.2, len(core_idx)) * np.log(4.0)
s1[medium_idx] = rng.beta(1.8, 3.0, len(medium_idx)) * np.log(4.0) * 0.5
s1[tail_idx] = np.sort(rng.exponential(0.03, len(tail_idx)))[::-1]

# ── Mutual information matrix ──────────────────────────────────────
mi = np.zeros((N, N))

# 1) Five intra-core clusters of ~10 orbitals each with strong MI
cluster_size = len(core_idx) // 5
for k in range(5):
    cl = core_idx[k * cluster_size : (k + 1) * cluster_size]
    for ii, i in enumerate(cl):
        for j in cl[ii + 1 :]:
            val = rng.beta(3, 1.5) * np.log(16.0) * 0.55
            mi[i, j] = mi[j, i] = val

# 2) Sparse inter-cluster core links
for ii, i in enumerate(core_idx):
    for j in core_idx[ii + 1 :]:
        if mi[i, j] == 0 and rng.random() < 0.15:
            val = rng.exponential(0.08)
            mi[i, j] = mi[j, i] = val

# 3) Medium orbitals: moderate MI to a few core and medium neighbours
for i in medium_idx:
    n_core_links = rng.integers(1, 4)
    targets = rng.choice(core_idx, size=n_core_links, replace=False)
    for j in targets:
        val = rng.exponential(0.12)
        mi[i, j] = mi[j, i] = val
    n_med_links = rng.integers(0, 3)
    others = rng.choice(
        medium_idx[medium_idx != i],
        size=min(n_med_links, len(medium_idx) - 1),
        replace=False,
    )
    for j in others:
        if mi[i, j] == 0:
            val = rng.exponential(0.06)
            mi[i, j] = mi[j, i] = val

# 4) Tail orbitals: very sparse, weak links to core or medium
for i in tail_idx:
    if rng.random() < 0.12:
        pool = np.concatenate([core_idx, medium_idx])
        j = rng.choice(pool)
        val = rng.exponential(0.02)
        mi[i, j] = mi[j, i] = val

np.fill_diagonal(mi, 0.0)

print(f"{N} orbitals, {N_CORE} selected (highlighted)")
print(f"Core indices (first 10): {core_idx[:10].tolist()} ...")
print(f"s1 range: {s1.min():.4f} – {s1.max():.4f}")
print(f"MI range: {mi[mi > 0].min():.4f} – {mi.max():.4f}")
print(f"Non-zero MI pairs: {(mi > 0).sum() // 2}")

## 2 — Display the interactive widget

The `OrbitalEntanglement` widget renders as an SVG chord diagram
directly in the notebook output.  Arc length encodes single-orbital
entropy; chord thickness encodes mutual information.  The 50 core
orbitals are highlighted with a dark outline.

In [ ]:
from qsharp_widgets import OrbitalEntanglement

widget = OrbitalEntanglement(
    s1_entropies=s1.tolist(),
    mutual_information=mi.tolist(),
    labels=[str(i) for i in range(N)],
    selected_indices=core_idx.tolist(),
    title=f"Synthetic Orbital Entanglement — {N} orbitals ({N_CORE} core)",
    group_selected=True,
    gap_deg=0.6,
    arc_width=0.05,
    mi_threshold=0.01,
    width=800,
    height=880,
)
widget

## 3 — Export as SVG (light & dark mode)

`export_svg()` renders the diagram server-side via Node.js — the same
Preact component used by the interactive widget — so fonts and colours
are deterministic regardless of viewer.

Use `dark_mode=True` for light text on a dark background, or
`dark_mode=False` (default) for dark text on a transparent background.

In [ ]:
from pathlib import Path

# Light mode (default)
light_path = Path("orbital_entanglement_light.svg")
saved = widget.export_svg(path=light_path, dark_mode=False)
print(f"Light SVG → {saved}  ({light_path.stat().st_size / 1024:.1f} KB)")

# Dark mode
dark_path = Path("orbital_entanglement_dark.svg")
saved = widget.export_svg(path=dark_path, dark_mode=True)
print(f"Dark  SVG → {saved}  ({dark_path.stat().st_size / 1024:.1f} KB)")

## 4 — Display the exported SVGs inline

Render both the light and dark variants to verify fonts, text
colour, and background are baked into the SVG correctly.

In [ ]:
from IPython.display import SVG, display, HTML

display(HTML("<h3>Light mode</h3>"))
display(SVG(filename=str(light_path)))

display(HTML("<h3>Dark mode</h3>"))
display(SVG(filename=str(dark_path)))